# 19 Batch presets

This notebook gives you practical preset modes for running the pipeline.
It reuses the orchestration logic and exports preset-specific status/plan tables.


In [ ]:
from pathlib import Path
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from src.batch_presets import (
    build_preset_plan,
    build_preset_status,
    export_preset_tables,
    list_presets,
    summarize_preset,
)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
PRESET_NAME = 'review_only'

# Valid options:
# review_only
# canonicalize_unresolved
# ocr_rescue
# promote_to_execution
# apply_small_live_batch
# rollback_last_batch


In [ ]:
presets_df = list_presets()
display(presets_df)


In [ ]:
summary_df = summarize_preset(OUTPUT_DIR, PRESET_NAME)
status_df = build_preset_status(OUTPUT_DIR, PRESET_NAME)
plan_df = build_preset_plan(OUTPUT_DIR, PRESET_NAME)

display(summary_df)
display(status_df)
display(plan_df)


In [ ]:
next_notebook = summary_df.iloc[0]['next_notebook'] if not summary_df.empty else None
if pd.isna(next_notebook) or next_notebook is None:
    print('All stages for this preset already have outputs.')
else:
    print('Recommended next notebook for preset', PRESET_NAME + ':', next_notebook)


In [ ]:
summary_path, status_path, plan_path = export_preset_tables(summary_df, status_df, plan_df, OUTPUT_DIR)
print('Wrote:', summary_path)
print('Wrote:', status_path)
print('Wrote:', plan_path)
